# Day 09. Exercise 02
# Metrics

## 0. Imports

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from joblib import dump
import numpy as np

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [3]:
df = pd.read_csv('../../data/day-of-week-not-scaled.csv')

old_df = pd.read_csv('../../data/dayofweek.csv')
df['dayofweek'] = old_df['dayofweek']

X = df.drop('dayofweek', axis=1)
y = df['dayofweek']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)


## 2. SVM

1. Use the best parameters from the previous exercise and train the model of SVM.
2. You need to calculate `accuracy`, `precision`, `recall`, `ROC AUC`.

 - `precision` and `recall` should be calculated for each class (use `average='weighted'`)
 - `ROC AUC` should be calculated for each class against any other class (all possible pairwise combinations) and then weighted average should be applied for the final metric
 - the code in the cell should display the result as below:

```
accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878
```

In [4]:
svc = SVC(random_state=21,
          kernel='rbf',
          gamma='auto',
          C=10.0,
          class_weight='balanced',
          probability=True)
svc.fit(X_train, y_train)
predict = svc.predict(X_test)
predict_proba = svc.predict_proba(X_test)

In [5]:
acc = accuracy_score(y_test, predict)
prec = precision_score(y_test, predict, average="weighted")
rec = recall_score(y_test, predict, average="weighted")
auc = roc_auc_score(y_test, predict_proba, multi_class="ovr")

for name, value in [
    ("accuracy", acc),
    ("precision", prec),
    ("recall", rec),
    ("roc_auc", auc),
]:
    print(f"{name} is {value:.5}")

accuracy is 0.88757
precision is 0.88826
recall is 0.88757
roc_auc is 0.97731


## 3. Decision tree

1. The same task for decision tree

In [6]:
tree = DecisionTreeClassifier(random_state=21,
                            class_weight=None,
                            criterion='gini',
                            max_depth=25,
                            )
tree.fit(X_train, y_train)
predict = tree.predict(X_test)
predict_proba = tree.predict_proba(X_test)

In [7]:
acc = accuracy_score(y_test, predict)
prec = precision_score(y_test, predict, average="weighted")
rec = recall_score(y_test, predict, average="weighted")
auc = roc_auc_score(y_test, predict_proba, multi_class="ovr")

for name, value in [
    ("accuracy", acc),
    ("precision", prec),
    ("recall", rec),
    ("roc_auc", auc),
]:
    print(f"{name} is {value:.5}")

accuracy is 0.86686
precision is 0.87097
recall is 0.86686
roc_auc is 0.91579


## 4. Random forest

1. The same task for random forest.

In [8]:
forest = RandomForestClassifier(random_state=21,
                            n_estimators=100,
                            max_depth=49, 
                            class_weight=None,
                            criterion='gini')
forest.fit(X_train, y_train)
predict = forest.predict(X_test)
predict_proba = forest.predict_proba(X_test)

In [9]:
acc = accuracy_score(y_test, predict)
prec = precision_score(y_test, predict, average="weighted")
rec = recall_score(y_test, predict, average="weighted")
auc = roc_auc_score(y_test, predict_proba, multi_class="ovr")

for name, value in [
    ("accuracy", acc),
    ("precision", prec),
    ("recall", rec),
    ("roc_auc", auc),
]:
    print(f"{name} is {value:.5}")

accuracy is 0.93787
precision is 0.93912
recall is 0.93787
roc_auc is 0.98841


## 5. Predictions

1. Choose the best model.
2. Analyze: for which `weekday` your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which `labname` and for which `users`.
3. Save the model.

In [10]:
df_pred = pd.DataFrame({
    'target': np.array(y_test),
    'prediction': predict
})

df_pred['mistake'] = df_pred['prediction'] != df_pred['target']

res = df_pred.groupby('target')['mistake'].mean() * 100
res

target
0    25.925926
1     5.454545
2     6.666667
3     2.500000
4    14.285714
5     5.555556
6     1.408451
Name: mistake, dtype: float64

In [11]:
dump(forest, '02_metrics.joblib')

['02_metrics.joblib']

## 6. Function

1. Write a function that takes a list of different models and a corresponding list of parameters (dicts) and returns a dict that contains all the 4 metrics for each model.

In [12]:
def get_metrics(models, params):
    res = {}
    
    for model_class, model_params in zip(models, params):

        model_name = model_class.__name__
        
        model = model_class(**model_params)
        model.fit(X_train, y_train)
        
        predict = model.predict(X_test)
        predict_proba = model.predict_proba(X_test)
        
        acc = accuracy_score(y_test, predict)
        prec = precision_score(y_test, predict, average="weighted")
        rec = recall_score(y_test, predict, average="weighted")
        auc = roc_auc_score(y_test, predict_proba, multi_class="ovr")
        
        res[model_name] = {
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "roc_auc": auc
        }
        
    return res

models = (SVC, DecisionTreeClassifier, RandomForestClassifier)
params = [{'random_state':21,
          'kernel':'rbf',
          'gamma':'auto',
          'C':10.0,
          'class_weight':'balanced',
          'probability':True},
          {'random_state':21,
            'class_weight':None,
            'criterion':'gini',
            'max_depth':25}]
metrics = get_metrics(models, params)
metrics
print(metrics)

{'SVC': {'accuracy': 0.8875739644970414, 'precision': 0.8882581396595722, 'recall': 0.8875739644970414, 'roc_auc': 0.9773098183794965}, 'DecisionTreeClassifier': {'accuracy': 0.8668639053254438, 'precision': 0.8709700321395994, 'recall': 0.8668639053254438, 'roc_auc': 0.9157852752779444}}
